# 166 — Sesgo, fairness y grupos afectados

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Una auditoría de fairness debe ser reproducible porque sus conclusiones tienen
consecuencias legales y sobre personas: el contrato (kind + evidence) fija los números medidos y el
protocolo, de modo que la disparidad reportada pueda re-verificarse y contrastarse.


In [ ]:
result = run_lab("evaluation", seed=166)
assert result["kind"] == "evaluation"
assert result["evidence"]
show(result)


**Ejercicio 2.**

```text
tasa X = 100/400 = 0.250
tasa Y = 45/300  = 0.150   <- desfavorecido
DI = 0.150 / 0.250 = 0.60   ->  < 0.80  ->  indicio de impacto adverso contra Y
```

**Ejercicio 3.**

```text
TPR X = 96/120 = 0.80
TPR Y = 52/80  = 0.65
```

No se cumple igualdad de oportunidad: entre los realmente aptos, Y es seleccionado con menor
frecuencia (0.65 vs 0.80). El sistema falla a la vez el disparate impact y la igualdad de oportunidad.

**Ejercicio 4 (esquema).** Como el score se usa directamente como probabilidad de impago para fijar
precio, la **calibración por grupo** es lo más defendible: un score de 0.1 debe significar el mismo
riesgo real en cada grupo, o el precio sería injusto e inexacto. El sacrificio, por Kleinberg, es
que con tasas base distintas no podrás además igualar FPR y FNR entre grupos: quienes son
erróneamente penalizados o beneficiados no se distribuirán igual. La elección debe documentarse.


In [ ]:
# Verificación numérica
seleccion = {"X": (100, 400), "Y": (45, 300)}
aptos = {"X": (96, 120), "Y": (52, 80)}
tasas = {k: s / n for k, (s, n) in seleccion.items()}
desfav = min(tasas, key=tasas.get)
fav = max(tasas, key=tasas.get)
DI = tasas[desfav] / tasas[fav]
tpr = {k: s / n for k, (s, n) in aptos.items()}
print(f"tasas={ {k: round(v,3) for k,v in tasas.items()} } desfavorecido={desfav}")
print(f"DI={DI:.2f} (regla 80%: {'FALLA' if DI < 0.80 else 'ok'})")
print(f"TPR={ {k: round(v,2) for k,v in tpr.items()} }")
assert round(DI, 2) == 0.60 and tpr["X"] == 0.80 and tpr["Y"] == 0.65


## Reflexión (guía)

1. Los **datos**: sesgo histórico, de medición y de representación. Cambiar de modelo sin corregir
   los datos suele preservar el sesgo.
2. Salvo predicción perfecta o tasas base iguales, no se pueden cumplir a la vez calibración por
   grupo e igualdad de las tasas de falsos positivos y falsos negativos.
3. Porque es un umbral legal orientativo sobre una sola métrica (tasa de positivos); puede cumplirse
   y aun así haber disparidad de TPR/FPR o de calibración.
